# Vỡ đập một chiều — Nhóm 7
**Mô hình, mô phỏng tương tác và kiểm chứng số**

Notebook đi từ nghiệm Stoker đến hai phương pháp thể tích hữu hạn Rusanov và HLL, rồi kiểm tra sai số, cân bằng và ảnh hưởng của tham số.

**Cách dùng:** chọn **Restart Kernel and Run All Cells**, đợi phần tính toán hoàn tất, rồi nhấn **▶ Phát** một lần. Thanh thời gian chạy tự động; có thể tạm dừng, kéo đến thời điểm cần xem, đổi phương pháp hoặc phát lại. Muốn thử cấu hình khác, sửa ô tham số và chạy lại toàn bộ.

Ở cấu hình ban đầu, thời gian vật lý là **0–6 s** với **121 trạng thái**; một lượt phát kéo dài khoảng **10 s** trên màn hình. Mô phỏng dừng ở giai đoạn sóng đang phát triển, không chạy đến khi mặt nước phẳng.

## 1. Mô hình và phương pháp

Kênh có đáy phẳng, bề rộng không đổi, không ma sát; nước ban đầu đứng yên, với $h_L>h_R>0$. Độ sâu $h$ và lưu lượng đơn vị $q=hu$ thỏa

$$\mathbf U_t+\mathbf F(\mathbf U)_x=0,\qquad
\mathbf U=\begin{pmatrix}h\\q\end{pmatrix},\qquad
\mathbf F(\mathbf U)=\begin{pmatrix}q\\q^2/h+gh^2/2\end{pmatrix}.$$

Bộ giải lưu **trung bình ô** của $h,q$. Sau khi tính thông lượng ở các mặt ô, toàn bộ các ô được cập nhật đồng thời:

$$\bar{\mathbf U}_i^{n+1}=\bar{\mathbf U}_i^n-
\frac{\Delta t_n}{\Delta x}\left(\widehat{\mathbf F}_{i+1/2}^n-
\widehat{\mathbf F}_{i-1/2}^n\right),\qquad
\Delta t_n=\min\left(C_{\mathrm{CFL}}\frac{\Delta x}{a_{\max}},T_k-t^n\right).$$

$T_k$ là mốc lưu kế tiếp; bước cuối được rút ngắn để đến đúng mốc này. Ở mỗi biên, ô ảo sao chép trạng thái của ô trong cùng phía. Vì vậy, nhật ký cân bằng phải tính cả thông lượng qua hai biên.

Rusanov dùng một tốc độ $a=\max(|u_L|+\sqrt{gh_L},|u_R|+\sqrt{gh_R})$:

$$\widehat{\mathbf F}_{\mathrm{Rus}}=\tfrac12(\mathbf F_L+\mathbf F_R)
-\tfrac a2(\mathbf U_R-\mathbf U_L).$$

HLL dùng hai tốc độ $s_L=\min(0,u_L-c_L,u_R-c_R)$, $s_R=\max(0,u_L+c_L,u_R+c_R)$:

$$\widehat{\mathbf F}_{\mathrm{HLL}}=
\frac{s_R\mathbf F_L-s_L\mathbf F_R+s_Ls_R(\mathbf U_R-\mathbf U_L)}{s_R-s_L}.$$

Ở các công thức thông lượng, $L,R$ chỉ hai phía của **mặt ô đang xét**. Đây là lựa chọn tốc độ từ hai trạng thái, không phải biến thể Einfeldt/Roe. Với sơ đồ đang dùng và độ sâu dương, CFL thực tế không vượt $1/2$ là điều kiện đủ giữ tính dương; chương trình dừng nếu gặp trạng thái không hợp lệ, không tự bổ sung nước.

In [ ]:
from pathlib import Path
import csv, hashlib, html, json, math, platform, sys, time
import numpy as np
import matplotlib.pyplot as plt
import plotly
from IPython.display import HTML, Markdown, display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'dambreak/interactive.py').is_file()
             and (p / 'bao-cao-nhom-7.ipynb').is_file()), None)
if ROOT is None:
    raise RuntimeError('Hãy mở Jupyter từ thư mục chứa notebook và thư mục dambreak/.')
sys.path.insert(0, str(ROOT))
from dambreak.exact import StokerProblem
from dambreak.solver import FVConfig
from dambreak.experiments import stoker_run, conservation_summary
from dambreak.interactive import run_simulation, make_figure, display_figure, save_figure
from dambreak.style import apply_style, COLORS
apply_style()

plt.rcParams['font.family'] = 'DejaVu Sans'
OUT = ROOT / 'outputs'; OUT.mkdir(exist_ok=True)


integrity = json.loads((ROOT / 'checks/source-integrity.json').read_text())
for group in ('source_sha256', 'data_sha256'):
    for name, expected in integrity[group].items():
        actual = hashlib.sha256((ROOT / name).read_bytes()).hexdigest()
        if actual != expected:
            raise RuntimeError(f'Tệp đã thay đổi so với bản được kiểm tra: {name}')

STYLE = '<style>.mhmp-table{border-collapse:collapse;font-size:14px}.mhmp-table th,.mhmp-table td{padding:7px 12px;border-bottom:1px solid #ddd;text-align:right}.mhmp-table th{color:#a52a2a}.mhmp-table th:first-child,.mhmp-table td:first-child{text-align:left}</style>'
def table(headers, rows):
    head = ''.join(f'<th>{html.escape(str(x))}</th>' for x in headers)
    body = ''.join('<tr>'+''.join(f'<td>{html.escape(str(x))}</td>' for x in row)+'</tr>' for row in rows)
    display(HTML(STYLE + '<table class="mhmp-table"><thead><tr>'+head+'</tr></thead><tbody>'+body+'</tbody></table>'))
def vi(x, digits=3):
    return f'{float(x):.{digits}f}'.replace('.', ',')
def sci(x):
    return f'{float(x):.3e}'.replace('.', ',')
def read_csv(path):
    with Path(path).open(newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))
def finish_figure(fig, name):
    fig.savefig(OUT / (name + '.png'), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

## 2. Chọn tham số và chạy mô phỏng

Độ sâu được nhập theo **mm**, sau đó đổi sang mét trước khi giải. Miền tính giữ $[-5,5]$ m, vị trí đập $x_d=0$ và $g=9{,}81$ m/s². Chọn $0<C_{\mathrm{CFL}}\leq0{,}5$. Thời điểm cuối phải nằm trước lúc sóng chính xác chạm biên; chương trình kiểm tra điều này.

`FRAMES` là số trạng thái lưu để xem, **không phải** số bước cập nhật PDE. `PLAYBACK_SECONDS` chỉ quyết định tốc độ phát.

In [ ]:
# Ô tham số duy nhất cần sửa khi thử bài toán khác.
N = 400
H_LEFT_MM = 5.0
H_RIGHT_MM = 1.0
CFL = 0.45
T_FINAL = 6.0
FRAMES = 121
PLAYBACK_SECONDS = 10.0

# Các giá trị dùng ở phần khảo sát; các tham số còn lại giữ theo ô này.
N_SWEEP = (100, 200, 400, 800)
CFL_SWEEP = (0.20, 0.30, 0.45)
DEPTH_RATIOS = (0.05, 0.20, 0.50)
DEPTH_SCALE_FACTORS = (0.5, 1.0, 2.0)
RUN_FULL_REPORT_STUDY = False  # Bật chỉ khi muốn chạy lại đủ 42 + 84 lượt của báo cáo.

# Báo lỗi sớm nếu danh sách khảo sát bị nhập sai.
if not (np.isfinite(PLAYBACK_SECONDS) and PLAYBACK_SECONDS > 0):
    raise ValueError('Thời gian phát phải hữu hạn và dương.')
if (len(N_SWEEP) < 2 or any(isinstance(x,bool) or not isinstance(x,(int,np.integer)) or x<2 for x in N_SWEEP)
        or any(b<=a for a,b in zip(N_SWEEP,N_SWEEP[1:]))):
    raise ValueError('N_SWEEP cần ít nhất hai số nguyên >= 2, tăng nghiêm ngặt.')
for label,values,upper,strict in (('CFL_SWEEP',CFL_SWEEP,.5,False),
                               ('DEPTH_RATIOS',DEPTH_RATIOS,1.,True),
                               ('DEPTH_SCALE_FACTORS',DEPTH_SCALE_FACTORS,float('inf'),True)):
    if (len(values)<2 or any(not np.isfinite(x) or x<=0 or (x>=upper if strict else x>upper) for x in values)
            or any(b<=a for a,b in zip(values,values[1:]))):
        raise ValueError(f'{label}: cần ít nhất hai giá trị dương, hữu hạn, tăng nghiêm ngặt; kiểm tra giới hạn cho phép.')


In [ ]:
started = time.perf_counter()
demo = run_simulation(cells=N, h_left_mm=H_LEFT_MM, h_right_mm=H_RIGHT_MM,
                      cfl=CFL, final_time=T_FINAL, frames=FRAMES)
problem = demo.problem
boundary_time = problem.first_boundary_time(-5.0, 5.0)
display(Markdown(
    f'**Cấu hình đang chạy:** N = {N}, Δx = {vi(demo.first.config.dx, 4)} m; '
    f'hL = {vi(H_LEFT_MM)} mm, hR = {vi(H_RIGHT_MM)} mm; CFL = {vi(CFL, 2)}. '
    f'Thời gian vật lý: 0–{vi(T_FINAL, 2)} s, {FRAMES} trạng thái; '
    f'thời điểm sóng chạm biên đầu tiên: {vi(boundary_time, 3)} s. '
    f'Tính xong hai phương pháp trong {vi(time.perf_counter()-started, 2)} s trên máy hiện tại.'))

# Chọn thời lượng đủ ngắn để giữ toàn bộ FRAMES trong danh sách phát.
figure = make_figure(demo, frame_duration_ms=1000*PLAYBACK_SECONDS/FRAMES,
                     playback_seconds=PLAYBACK_SECONDS)
assert figure.layout.meta['displayed_frame_count'] == FRAMES
html_path = save_figure(figure, OUT / 'mo-phong-vo-dap.html')
display_figure(figure)

**Đọc đồ thị:** đường đen là trung bình ô Stoker, đường đỏ là HLL, đường cam nét đứt là Rusanov. Ở biểu đồ tương tác, $h$ dùng **mm** và $q$ dùng **mm²/s**; $1\,\mathrm{mm^2/s}=10^{-6}\,\mathrm{m^2/s}$. Các biểu đồ tĩnh phía dưới ghi đơn vị riêng trên trục.

Nhấn **Phát** một lần để xem cả diễn biến. **Tạm dừng** giữ thời điểm hiện tại; kéo thanh thời gian để chọn một trạng thái đã tính; **Về đầu** trở lại $t=0$. Nút HLL/Rusanov chỉ thay đổi đường được hiển thị. Mục **Toàn miền** cho thấy phần nước chưa chịu ảnh hưởng.

Sau khi chạy ô mô phỏng, mở tệp `outputs/mo-phong-vo-dap.html` nếu cần trình chiếu ngoài Jupyter.

In [ ]:
table(['Thông lượng', 'e₁,h (%)', 'e₁,q (%)', 'Số bước PDE', 'h nhỏ nhất (mm)'], [
    [method.upper(), vi(100*demo.error_rows[method][-1]['e1_h'], 4),
     vi(100*demo.error_rows[method][-1]['e1_q'], 4),
     conservation_summary(run)['steps'],
     vi(1000*conservation_summary(run)['min_depth_all_steps'], 4)]
    for method, run in demo.results.items()])
display(Markdown(f'Bảng trên dùng **lịch lưu hoạt ảnh**, với {FRAMES} mốc trong '
                 f'0–{vi(T_FINAL,2)} s. Sai số này không thay thế số liệu đo theo lịch lưu của báo cáo.'))

### Nghiệm Stoker cho biết điều gì?

Độ sâu trung gian $h_*$ là nghiệm duy nhất trong $(h_R,h_L)$ của

$$2\left(\sqrt{gh_L}-\sqrt{gh_*}\right)
=(h_*-h_R)\sqrt{\frac g2\left(\frac1{h_*}+\frac1{h_R}\right)}.$$

Sau khi tìm $h_*$ bằng chia đôi, $u_*=2(\sqrt{gh_L}-\sqrt{gh_*})$ và tốc độ sốc
$s=h_*u_* /(h_*-h_R)$. Bốn vùng từ trái sang phải là nước thượng lưu, quạt giãn, trạng thái trung gian và nước hạ lưu. Tốc độ đầu quạt là $-\sqrt{gh_L}$, đuôi quạt là $u_*-\sqrt{gh_*}$; đuôi quạt có thể nằm bên trái hoặc bên phải vị trí đập tùy tỉ số độ sâu.

Nghiệm tham chiếu cũng phải lấy **trung bình trên từng ô**. Mỗi ô được chia tại các mép sóng nằm bên trong nó; trong quạt dùng Gauss 3 điểm, chính xác với các đa thức của $h,q$ trong số học chính xác. Sai số làm tròn và sai số tìm $h_*$ vẫn còn ở mức rất nhỏ.

In [ ]:
table(['Đại lượng Stoker', 'Giá trị'], [
    ['h* (mm)', vi(1000*problem.middle_depth, 6)],
    ['u* (m/s)', vi(problem.middle_velocity, 6)],
    ['Tốc độ đầu quạt (m/s)', vi(problem.head_speed, 6)],
    ['Tốc độ đuôi quạt (m/s)', vi(problem.tail_speed, 6)],
    ['Tốc độ sốc (m/s)', vi(problem.shock_speed, 6)],
    [f'Vị trí sốc tại t = {vi(T_FINAL,2)} s (m)', vi(problem.shock_speed*T_FINAL, 6)]])

## 3. Đo sai số và kiểm tra cân bằng

Phần này giải lại bài toán với bốn mốc $[0,T/3,2T/3,T]$; ở cấu hình chuẩn chúng là $[0,2,4,6]$ s, giống báo cáo. Giữ cùng lịch lưu cho hai thông lượng vì việc rút ngắn bước cuối mỗi khoảng có thể làm sai số thay đổi nhẹ.

Hai sai số được chuẩn hóa riêng:

$$e_{1,h}=\frac{\Delta x\sum_i|\bar h_i-\bar h_i^{\mathrm{ref}}|}{Lh_L},\qquad
 e_{1,q}=\frac{\Delta x\sum_i|\bar q_i-\bar q_i^{\mathrm{ref}}|}{Lh_L\sqrt{gh_L}},\quad L=10\,\mathrm m.$$

Đây là sai số tích phân trên toàn miền, không phải phần trăm sai số ở từng điểm. Vùng sốc vẫn có thể bị trải rộng dù sai số tích phân nhỏ.

In [ ]:
case_cache = {}
def compute_case(method, cells=N, cfl=CFL, h_left_mm=H_LEFT_MM,
                 h_right_mm=H_RIGHT_MM, final_time=T_FINAL):
    key = (method, int(cells), float(cfl), float(h_left_mm), float(h_right_mm), float(final_time))
    if key not in case_cache:
        p = StokerProblem(h_left=h_left_mm/1000, h_right=h_right_mm/1000)
        cfg = FVConfig(cells=cells, cfl=cfl, flux=method)
        times = [0.0, final_time/3, 2*final_time/3, final_time]
        result, summary = stoker_run(p, cfg, times)
        case_cache[key] = (p, result, summary)
    return case_cache[key]

METHODS = ('rusanov', 'hll')
base = {m: compute_case(m) for m in METHODS}
table(['Thông lượng', 'e₁,h (%)', 'e₁,q (%)', 'Số bước', 'CFL thực tế lớn nhất'], [
    [m.upper(), vi(100*s['errors'][-1]['e1_h'], 4), vi(100*s['errors'][-1]['e1_q'], 4),
     s['diagnostics']['steps'], vi(s['diagnostics']['max_actual_cfl'], 4)]
    for m, (p, r, s) in base.items()])

is_default = (N == 400 and H_LEFT_MM == 5.0 and H_RIGHT_MM == 1.0
              and CFL == 0.45 and T_FINAL == 6.0)
if is_default:
    saved = read_csv(ROOT / 'data/report-baseline/convergence.csv')
    for m, (p, result, summary) in base.items():
        old = next(row for row in saved if row['flux']==m and int(row['cells'])==N
                   and float(row['time'])==T_FINAL)
        for component in ('h','q'):
            np.testing.assert_allclose(summary['errors'][-1]['e1_'+component],
                                       float(old['e1_'+component]), rtol=2e-12, atol=2e-15)
    display(Markdown('**Đối chiếu đạt:** sai số của hai lượt chuẩn vừa tính khớp CSV của báo cáo trong dung sai kiểm tra.'))
else:
    display(Markdown('Đây là **cấu hình tự chọn**. Kết quả được tính mới; CSV của báo cáo vẫn giữ cấu hình gốc và được xem riêng ở phần 5.'))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True, constrained_layout=True)
for m, (p, result, summary) in base.items():
    exact = p.cell_averages(result.edges, T_FINAL)
    for j, scale in ((0,1000),(1,1e4)):
        axes[j].plot(result.centers, result.states[-1,:,j]*scale,
                     color=COLORS[m], label=m.upper(), linestyle='-' if m=='hll' else '--')
        if m == METHODS[0]:
            axes[j].stairs(exact[:,j]*scale, result.edges, color='black', linewidth=1.1,
                           label='Stoker: trung bình ô')
for ax in axes:
    ax.grid(); ax.legend(ncol=3, fontsize=9)
    head, _, shock = problem.wave_positions(T_FINAL)
    pad = max(.2*(shock-head), 3*demo.first.config.dx)
    ax.set_xlim(max(-5,head-pad), min(5,shock+pad))
axes[0].set_ylabel('Độ sâu h̄ (mm)')
axes[1].set_ylabel('Lưu lượng q̄ (10⁻⁴ m²/s)'); axes[1].set_xlabel('Vị trí x (m)')
axes[0].set_title(f'Nghiệm tại t = {vi(T_FINAL,2)} s · N = {N} · CFL = {vi(CFL,2)}')
finish_figure(fig, 'nghiem-so-va-stoker')

Tổng độ sâu $M=\Delta x\sum_i\bar h_i$ có đơn vị m², tương ứng thể tích trên một đơn vị bề rộng. Tổng $P=\Delta x\sum_i\bar q_i$ có đơn vị m³/s, tương ứng động lượng sau khi chia cho khối lượng riêng và bề rộng. Với cùng thông lượng dùng để cập nhật nghiệm,

$$R_M^n=M^n-M^0-\sum_{k<n}\Delta t_k(\widehat F_{1/2,h}^k-\widehat F_{N+1/2,h}^k),$$

và $R_P$ được tính tương tự bằng thành phần thông lượng động lượng. Trong bài vỡ đập, $P$ tăng do chênh lệch áp lực ở hai biên; **động lượng không cần giữ hằng**. Điều cần kiểm tra là phần dư sau khi tính đóng góp của biên nhỏ.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
rows=[]
for m, (p, result, summary) in base.items():
    hist=result.history; diag=summary['diagnostics']
    for ax,key in zip(axes,('mass_residual','momentum_residual')):
        ax.plot([r['time'] for r in hist], [r[key] for r in hist], color=COLORS[m], label=m.upper())
        ax.grid(); ax.set_xlabel('Thời gian (s)'); ax.legend(); ax.ticklabel_format(axis='y',style='sci',scilimits=(0,0))
    rows.append([m.upper(), vi(1000*diag['min_depth_all_steps'],5),
                 sci(diag['max_abs_mass_residual']), sci(diag['max_abs_momentum_residual'])])
axes[0].set_title('Phần dư khối lượng');axes[0].set_ylabel('Rₘ (m²)')
axes[1].set_title('Phần dư động lượng');axes[1].set_ylabel('Rₚ (m³/s)')
finish_figure(fig,'can-bang')
table(['Thông lượng','h nhỏ nhất (mm)','max |Rₘ| (m²)','max |Rₚ| (m³/s)'],rows)

## 4. Thử thay đổi tham số

Các khảo sát sau **chạy lại bộ giải trong phiên notebook này**. Mỗi lần chỉ đổi nhóm tham số đang xét; các tham số khác lấy từ ô cấu hình. Mọi cặp HLL/Rusanov dùng cùng lưới và lịch lưu. Các lượt trùng cấu hình được dùng lại để tránh tính lặp.

### 4.1. Số ô $N$

Tăng $N$ làm $\Delta x$ nhỏ hơn và cho phép mô tả sốc trên một vùng hẹp hơn. Bậc quan sát giữa hai lưới là $p=\log(e_\mathrm{thô}/e_\mathrm{mịn})/\log(N_\mathrm{mịn}/N_\mathrm{thô})$. Do nghiệm có gián đoạn, không nên đòi bậc một trên bài này; báo cáo kiểm tra bậc của phương pháp bằng một bài sóng trơn riêng.

In [ ]:
from matplotlib.ticker import NullLocator
grid_results={m:[compute_case(m,cells=int(n)) for n in N_SWEEP] for m in METHODS}
fig,axes=plt.subplots(1,2,figsize=(10,3.8),constrained_layout=True)
rows=[]
for m, cases in grid_results.items():
    prev=None
    for n,(p,result,s) in zip(N_SWEEP,cases):
        e=s['errors'][-1]
        ph='—' if prev is None else vi(math.log(prev[1]/e['e1_h'])/math.log(n/prev[0]),3)
        pq='—' if prev is None else vi(math.log(prev[2]/e['e1_q'])/math.log(n/prev[0]),3)
        rows.append([m.upper(),n,vi(100*e['e1_h'],4),ph,vi(100*e['e1_q'],4),pq])
        prev=(n,e['e1_h'],e['e1_q'])
    for ax,comp in zip(axes,('h','q')):
        ax.loglog(N_SWEEP,[100*s['errors'][-1]['e1_'+comp] for p,r,s in cases],
                  'o-',color=COLORS[m],label=m.upper())
        ax.set_xticks(N_SWEEP, [str(n) for n in N_SWEEP]);ax.xaxis.set_minor_locator(NullLocator())
        ax.set_xlabel('Số ô N');ax.set_ylabel(f'e₁,{comp} (%)');ax.grid(which='both');ax.legend()
axes[0].set_title('Sai số độ sâu');axes[1].set_title('Sai số lưu lượng')
finish_figure(fig,'anh-huong-luoi')
table(['Thông lượng','N','e₁,h (%)','pₕ','e₁,q (%)','p_q'],rows)

### 4.2. Hệ số CFL

CFL nhỏ làm bước thời gian ngắn hơn, nên thường cần nhiều bước hơn để đến cùng thời điểm cuối. Tuy nhiên, bước thời gian nhỏ hơn không bảo đảm sai số tổng nhỏ hơn: khuếch tán số của sơ đồ đầy đủ còn phụ thuộc quan hệ giữa $\Delta t$ và $\Delta x$. Bảng và hình dưới đây giúp kiểm tra điều đó trên cấu hình hiện tại. Số bước không phải là phép đo thời gian chạy.

In [ ]:
cfl_results={m:[compute_case(m,cfl=float(c)) for c in CFL_SWEEP] for m in METHODS}
fig,axes=plt.subplots(1,2,figsize=(10,3.7),constrained_layout=True)
rows=[]
for m,cases in cfl_results.items():
    axes[0].plot(CFL_SWEEP,[100*s['errors'][-1]['e1_h'] for p,r,s in cases],'o-',color=COLORS[m],label=m.upper())
    axes[1].plot(CFL_SWEEP,[s['diagnostics']['steps'] for p,r,s in cases],'o-',color=COLORS[m],label=m.upper())
    for c,(p,r,s) in zip(CFL_SWEEP,cases):
        rows.append([m.upper(),vi(c,2),vi(100*s['errors'][-1]['e1_h'],4),
                     vi(100*s['errors'][-1]['e1_q'],4),s['diagnostics']['steps']])
for ax in axes:ax.set_xlabel('CFL đặt trước');ax.grid();ax.legend()
axes[0].set_title('Sai số độ sâu');axes[0].set_ylabel('e₁,h (%)')
axes[1].set_title('Số bước cập nhật');axes[1].set_ylabel('Số bước PDE')
finish_figure(fig,'anh-huong-cfl')
table(['Thông lượng','CFL','e₁,h (%)','e₁,q (%)','Số bước'],rows)

### 4.3. Tỉ số độ sâu $r=h_R/h_L$

Ở đây giữ $h_L$ và thay $h_R$. Đây là các **bài toán vật lý khác nhau**, không phải một dãy làm mịn lưới. Đường liền màu biểu diễn HLL; đường đen nét đứt là Stoker tương ứng. Khi hạ lưu nông hơn, vận tốc trung gian tăng, nhưng lưu lượng $q_*=h_*u_*$ còn phụ thuộc $h_*$. Không thể suy ra xu hướng của lưu lượng chỉ từ vận tốc.

In [ ]:
ratio_results={m:[compute_case(m,h_right_mm=H_LEFT_MM*float(r)) for r in DEPTH_RATIOS] for m in METHODS}
fig,axes=plt.subplots(2,1,figsize=(10,6),sharex=True,constrained_layout=True)
palette=['#CC0000','#CC6600','#28648C','#7353A6','#3C763D']
rows=[]
for idx,(ratio,(p,result,s)) in enumerate(zip(DEPTH_RATIOS,ratio_results['hll'])):
    ref=p.cell_averages(result.edges,T_FINAL)
    for j,scale in ((0,1000),(1,1e4)):
        axes[j].plot(result.centers,result.states[-1,:,j]*scale,color=palette[idx%len(palette)],label=f'HLL · r = {vi(ratio,2)}')
        axes[j].plot(result.centers,ref[:,j]*scale,'--',color='black',linewidth=.8,label='Stoker' if idx==0 else None)
for m,cases in ratio_results.items():
    for ratio,(p,result,s) in zip(DEPTH_RATIOS,cases):
        rows.append([m.upper(),vi(ratio,2),vi(p.middle_velocity,5),vi(p.middle_depth*p.middle_velocity*1e4,5),
                     vi(100*s['errors'][-1]['e1_h'],4),vi(100*s['errors'][-1]['e1_q'],4)])
for ax in axes:ax.grid();ax.legend(ncol=4,fontsize=8)
axes[0].set_title(f'Thay đổi tỉ số độ sâu · hL = {vi(H_LEFT_MM,2)} mm · t = {vi(T_FINAL,2)} s')
axes[0].set_ylabel('Độ sâu h̄ (mm)');axes[1].set_ylabel('Lưu lượng q̄ (10⁻⁴ m²/s)');axes[1].set_xlabel('Vị trí x (m)')
# Thu khung nhìn theo các sóng xa nhất trong nhóm đang xét.
left=min(p.head_speed*T_FINAL for p,r,s in ratio_results['hll'])
right=max(p.shock_speed*T_FINAL for p,r,s in ratio_results['hll'])
axes[0].set_xlim(max(-5,left-.2*(right-left)),min(5,right+.2*(right-left)))
finish_figure(fig,'anh-huong-ti-so-do-sau')
table(['Thông lượng','r','u* (m/s)','q* (10⁻⁴ m²/s)','e₁,h (%)','e₁,q (%)'],rows)

### 4.4. Cùng tỉ số, nhưng thay cả hai độ sâu

Nhân đồng thời $h_L,h_R$ với một hệ số giữ nguyên tỉ số $r$. Tốc độ đặc trưng tăng theo căn bậc hai của độ sâu. Vì vậy, tại cùng thời gian vật lý, sóng của trường hợp sâu hơn đã đi xa hơn.

Để kiểm tra tính đồng dạng, so sánh tại cùng $\tau=T\sqrt{gH}/L$ với $H=h_L$. Khi đổi $H$ thành $\alpha H$, thời điểm tương ứng là $T/\sqrt\alpha$. Phải đổi **toàn bộ lịch lưu** theo cùng hệ số. Khi đó các biến $h/H$ và $q/(H\sqrt{gH})$ có thể được so sánh trực tiếp.

In [ ]:
scaled={}
for mode in ('fixed','matched'):
    scaled[mode]={m:[compute_case(m,h_left_mm=H_LEFT_MM*a,h_right_mm=H_RIGHT_MM*a,
                                 final_time=T_FINAL if mode=='fixed' else T_FINAL/math.sqrt(a))
                     for a in DEPTH_SCALE_FACTORS] for m in METHODS}
fig,axes=plt.subplots(2,1,figsize=(10,6),sharex=True,constrained_layout=True)
for idx,a in enumerate(DEPTH_SCALE_FACTORS):
    for ax,mode in zip(axes,('fixed','matched')):
        p,result,s=scaled[mode]['hll'][idx]
        values=result.states[-1,:,0]*(1000 if mode=='fixed' else 1/p.h_left)
        ax.plot(result.centers,values,color=palette[idx%len(palette)],
                linestyle=('-', '--', ':')[idx%3],label=f'H = {vi(p.h_left*1000,2)} mm')
for ax in axes:ax.grid();ax.legend(ncol=3)
axes[0].set_title(f'HLL: cùng thời gian vật lý t = {vi(T_FINAL,2)} s');axes[0].set_ylabel('Độ sâu h̄ (mm)')
axes[1].set_title('HLL: cùng thời gian không thứ nguyên');axes[1].set_ylabel('Độ sâu chuẩn hóa h̄/H');axes[1].set_xlabel('Vị trí x (m)')
left=min(p.head_speed*r.times[-1] for mode in scaled.values() for p,r,s in mode['hll'])
right=max(p.shock_speed*r.times[-1] for mode in scaled.values() for p,r,s in mode['hll'])
axes[0].set_xlim(max(-5,left-.15*(right-left)),min(5,right+.15*(right-left)))
finish_figure(fig,'thang-do-sau')
rows=[]; similarity=[]
for mode in ('fixed','matched'):
    for m,cases in scaled[mode].items():
        for a,(p,r,s) in zip(DEPTH_SCALE_FACTORS,cases):
            rows.append(['Cùng t' if mode=='fixed' else 'Cùng τ',m.upper(),vi(p.h_left*1000,2),
                         vi(r.times[-1],3),vi(100*s['errors'][-1]['e1_h'],4)])
for m,cases in scaled['matched'].items():
    p0,r0,_=cases[0]; reference_scaled=r0.states[-1]/np.array([p0.h_left,p0.h_left*p0.c_left])
    delta=np.zeros(2)
    for p,r,s in cases[1:]:
        delta=np.maximum(delta,np.max(np.abs(r.states[-1]/np.array([p.h_left,p.h_left*p.c_left])-reference_scaled),axis=0))
    similarity.append([m.upper(),sci(delta[0]),sci(delta[1])])
table(['Chế độ','Thông lượng','H (mm)','T (s)','e₁,h (%)'],rows)
table(['So sánh cùng τ','max |Δ(h/H)|','max |Δ(q/(H√gH))|'],similarity)

## 5. Đối chiếu với bộ 84 lượt của báo cáo

Phần này **đọc dữ liệu đã lưu**, không chạy lại 84 lượt. Bộ khảo sát gồm 72 tổ hợp của hai thông lượng, bốn lưới, ba CFL và ba tỉ số độ sâu, cùng 12 lượt thay thang độ sâu. Tổng cộng có **80 cấu hình khác nhau** vì một số cấu hình chuẩn được lặp giữa các nhóm. Sửa ô tham số ở phần 2 không làm thay đổi các CSV này.

Hình dưới đây tổng hợp sai số độ sâu theo phần trăm ở $t=6$ s. Khi so sánh hai phương pháp, chọn cùng $N$, CFL và $r$. Màu dùng chung một thang cho cả sáu bảng nhỏ.

In [ ]:
saved_summary=json.loads((ROOT/'data/report-parameters/summary.json').read_text())
factorial=read_csv(ROOT/'data/report-parameters/factorial.csv')
all_saved=read_csv(ROOT/'data/report-parameters/all-runs.csv')
assert len(factorial)==72 and len(all_saved)==84
from matplotlib.colors import LogNorm
ns=saved_summary['grids']; cs=saved_summary['cfl_values']; ratios=saved_summary['depth_ratios']
values=np.array([100*float(row['e1_h']) for row in factorial])
fig,axes=plt.subplots(2,3,figsize=(10,6),sharex=True,sharey=True,constrained_layout=True)
for i,m in enumerate(METHODS):
    for j,ratio in enumerate(ratios):
        array=np.array([[100*float(next(row['e1_h'] for row in factorial
                    if row['flux']==m and int(row['cells'])==n
                    and float(row['requested_cfl'])==c and float(row['depth_ratio'])==ratio))
                    for c in cs] for n in ns])
        im=axes[i,j].imshow(array,cmap='OrRd',norm=LogNorm(values.min(),values.max()),aspect='auto')
        axes[i,j].set_title(f'{m.upper()} · r = {vi(ratio,2)}')
        axes[i,j].set_xticks(range(len(cs)),[vi(c,2) for c in cs]); axes[i,j].set_yticks(range(len(ns)),ns)
        for r in range(len(ns)):
            for c in range(len(cs)):
                axes[i,j].text(c,r,vi(array[r,c],3),ha='center',va='center',fontsize=8,
                               color='white' if array[r,c]>.9 else 'black')
        if i==1:axes[i,j].set_xlabel('CFL')
        if j==0:axes[i,j].set_ylabel('Số ô N')
fig.colorbar(im,ax=axes,label='e₁,h (%) · thang màu logarit',shrink=.85)
finish_figure(fig,'bao-cao-72-to-hop')
table(['Bộ dữ liệu đã lưu','Giá trị'],[
    ['Số lượt / số cấu hình khác nhau',f"{saved_summary['number_of_pde_runs']} / {saved_summary['number_of_unique_configurations']}"],
    ['Độ sâu nhỏ nhất (mm)',vi(1000*saved_summary['minimum_depth'],5)],
    ['Phần dư khối lượng lớn nhất (m²)',sci(saved_summary['maximum_mass_residual'])],
    ['Phần dư động lượng lớn nhất (m³/s)',sci(saved_summary['maximum_momentum_residual'])]])

## 6. Chạy lại toàn bộ khảo sát và lưu kết quả

`RUN_FULL_REPORT_STUDY=False` giữ phần này ở trạng thái tắt. Nếu bật rồi chạy lại, chương trình tái tạo **42 lượt cơ sở và 84 lượt bổ sung** với đúng cấu hình của báo cáo, ghi vào thư mục `outputs/full-report-rerun/`. Các tham số ở phần 2 chỉ điều khiển phần minh họa và khảo sát nhỏ; bộ báo cáo đầy đủ có cấu hình cố định trong mã nguồn.

In [ ]:
if RUN_FULL_REPORT_STUDY:
    from dambreak.experiments import run_study
    from parameter_study import run as run_parameter_study
    full_dir=OUT/'full-report-rerun'
    baseline_summary=run_study(full_dir/'baseline',make_plots=False)
    full_summary=run_parameter_study(full_dir/'parameters',full_dir/'baseline')
    assert baseline_summary['number_of_pde_runs']==42
    assert full_summary['number_of_pde_runs']==84
    display(Markdown('Đã chạy lại đủ **42 + 84 lượt**. Dữ liệu mới nằm trong `outputs/full-report-rerun/`.'))
else:
    display(Markdown('**Không chạy bộ đầy đủ trong phiên này.** Hình ở phần 5 dùng CSV đã lưu và kiểm tra dấu SHA-256.'))

# Lưu kết quả tóm tắt từ những lượt tính mới trong notebook.
computed=[]
for key,(p,result,summary) in case_cache.items():
    method,cells,cfl,hl,hr,tf=key
    computed.append({**summary['errors'][-1], **summary['diagnostics'],
                     'flux':method, 'cells':cells, 'cfl':cfl, 'h_left_mm':hl, 'h_right_mm':hr,
                     'final_time':tf, 'output_times':result.times.tolist()})
with (OUT/'khao-sat-trong-notebook.csv').open('w',newline='',encoding='utf-8') as stream:
    writer=csv.DictWriter(stream,fieldnames=list(computed[0]));writer.writeheader();writer.writerows(computed)
summary=dict(parameters=dict(N=N,h_left_mm=H_LEFT_MM,h_right_mm=H_RIGHT_MM,cfl=CFL,
             final_time=T_FINAL,frames=FRAMES,playback_seconds=PLAYBACK_SECONDS),
             demonstration_runs=2,report_schedule_runs=len(case_cache),
             saved_report_runs=84,full_report_rerun=RUN_FULL_REPORT_STUDY,
             all_depths_positive=all(s['diagnostics']['min_depth_all_steps']>0 for p,r,s in case_cache.values()),
             max_actual_cfl=max(s['diagnostics']['max_actual_cfl'] for p,r,s in case_cache.values()),
             max_abs_mass_residual=max(s['diagnostics']['max_abs_mass_residual'] for p,r,s in case_cache.values()),
             max_abs_momentum_residual=max(s['diagnostics']['max_abs_momentum_residual'] for p,r,s in case_cache.values()),
             displayed_frames=figure.layout.meta['displayed_frame_count'],
             simulated_time=figure.layout.meta['simulation_end_s'],
             versions=dict(python=platform.python_version(),numpy=np.__version__,plotly=plotly.__version__))
assert summary['all_depths_positive'] and summary['max_actual_cfl']<=0.5+1e-14
(OUT/'tom-tat-phien-chay.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
display(Markdown(f'Đã tính mới **2 lượt cho hoạt ảnh và {len(case_cache)} lượt theo lịch lưu thưa**. '
                 'Bảng tóm tắt, các hình PNG và bản HTML tương tác đã được lưu trong `outputs/`.'))

## Đọc kết quả

- **Lưới:** xem sai số giảm ra sao khi tăng $N$, đồng thời quan sát bề rộng vùng sốc bị làm trơn. Không dùng riêng một hình để kết luận về bậc hội tụ.
- **CFL:** so sánh sai số cùng với số bước. CFL nhỏ hơn không tự động cho kết quả tốt hơn ở cùng lưới.
- **Độ sâu:** thay $r$ làm đổi cấu trúc và biên độ sóng; thay cả hai độ sâu còn làm đổi thang thời gian. Phân biệt so sánh cùng $t$ với cùng $\tau$.
- **Cân bằng:** kiểm tra phần dư có thông lượng biên, thay vì yêu cầu động lượng giữ hằng.
- **Phạm vi:** các kết quả là kiểm chứng số cho mô hình một chiều, đáy phẳng, nền ướt và không ma sát. Chúng chưa phải đối chiếu với thí nghiệm vỡ đập thực tế.

### Tài liệu tham khảo

1. David I. Ketcheson, Randall J. LeVeque, and Mauricio J. del Razo (2020). *Riemann Problems and Jupyter Solutions: Theory and Approximate Solvers for Hyperbolic PDEs*. SIAM. [Shallow water chapter](https://www.clawpack.org/riemann_book/html/Shallow_water.html).
2. Randall J. LeVeque (2002). *Finite Volume Methods for Hyperbolic Problems*. Cambridge University Press. [DOI](https://doi.org/10.1017/CBO9780511791253).
3. Amiram Harten, Peter D. Lax, and Bram van Leer (1983). *On Upstream Differencing and Godunov-Type Schemes for Hyperbolic Conservation Laws*. SIAM Review, 25(1), 35–61. [DOI](https://doi.org/10.1137/1025002).

Mã cập nhật PDE nằm trong `dambreak/solver.py`; nghiệm Stoker trong `dambreak/exact.py`; bộ khảo sát đầy đủ trong `dambreak/experiments.py` và `parameter_study.py`.